In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Conditional Generation

The garment generator of that section and the sprite generators of that section take no request. Each returns a sample from the whole data distribution: to get a pair of trousers from the garment generator we can only draw again until one appears, and nothing lets us ask the sprite generator for a particular creature. Most image generators that people use take a request instead: a class, a caption, a sketch to complete. Within months of the original GAN, @Mirza.Osindero.2014 proposed the obvious repair: give the request to both players. The generator receives it as an extra input, and the critic judges whether a sample is a realistic instance of that request.

Is that one change enough? This section asks what the game then optimizes, how the critic should read the request, and how we can tell whether the generator obeys. The answers reuse the chapter's log-loss game, pairing loss and penalties, applied once per request. The running example is CIFAR-10: ask for a bird, and check that we get one. The check turns out to be the harder half, because a generator can obey every request and still return a single image per class.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## The Conditional Game

### The Optimal Conditional Critic

A condition specifies a property of the requested sample. It may be a class label, a caption embedding, or an entire image; our running example uses the ten CIFAR-10 class labels. The data consists of image--label pairs drawn from a joint distribution $p(x,c)$. The generator also receives the condition, so $x' = G(z,c)$, and varying $z$ at fixed $c$ samples from a conditional distribution $q(\cdot \mid c)$. We draw the generator's conditions from the data marginal $p(c)$, so the generated pairs follow $q(x,c)=p(c)\,q(x\mid c)$. The real and generated joint distributions then share the condition marginal and differ only in their conditional image distributions.

The critic scores sample--condition pairs, $D(x, c)$, and the game is the log-loss game of that section played on those pairs:

$$
V(D) \;=\; \mathbb{E}_{(x, c) \sim p}\big[\log \sigma(D(x, c))\big]
+ \mathbb{E}_{c \sim p(c),\; x' \sim q(\cdot \mid c)}\big[\log \sigma(-D(x', c))\big].
$$

In the unconditional game the optimal critic is the log density ratio $\log(p/q)$, and the value at that critic is $2\,\mathrm{JS}(p, q) - 2\log 2$. The pointwise argument behind both results works for any input space, so it applies with the pair $(x, c)$ in place of $x$. The optimal critic is the log ratio of the two joint densities, and their shared condition marginal cancels:

$$
D^\star(x, c)
\;=\; \log \frac{p(x, c)}{q(x, c)}
\;=\; \log \frac{p(c)\, p(x \mid c)}{p(c)\, q(x \mid c)}
\;=\; \log \frac{p(x \mid c)}{q(x \mid c)} .
$$

The optimal conditional critic is the log density ratio within the slice of data that the condition selects. For the bird condition it compares real and generated birds; images of other classes do not enter. Substituting this critic into the equation keeps the outer expectation over $c$, and each condition contributes the value of its own unconditional game:

$$
\max_D V(D)
\;=\; \mathbb{E}_{c}\Big[\, 2\, \mathrm{JS}\big(p(\cdot \mid c),\, q(\cdot \mid c)\big) \Big] - 2 \log 2 .
$$

Thus the conditional game is an average of per-condition games, weighted by how often each condition occurs. Its value reaches the minimum $-2\log 2$ exactly when $q(\cdot \mid c) = p(\cdot \mid c)$ for every condition that $p(c)$ weights: the generator must match the data within every slice.

### Stronger than Marginal Matching

This requirement is stronger than matching the distribution of images. Consider a generator that produces accurate CIFAR-10 images but shifts every label cyclically, so that $q(\cdot \mid c) = p(\cdot \mid c{+}1)$: a request for a bird returns a cat. All ten classes occur equally often, so the shift leaves the mixture over classes unchanged. The generator's distribution of images equals the data's, and the unconditional game sits at its minimum, $-2\log 2$; it cannot see the wrong labels. The conditional game compares each slice with its neighbor instead. If the classes had disjoint supports, every slice would contribute $\mathrm{JS} = \log 2$, and the equation would equal $0$, the largest value the game can take. CIFAR-10 classes overlap a little, so the value sits just below that maximum: a perfect image generator that fails every request is close to the worst conditional generator (Exercise 1 computes both values for two Gaussian slices).

The weighting by $p(c)$ has a cost for imbalanced data. A rare condition contributes little to the average, so the generator can serve it poorly at a small cost to the objective (Exercise 6).

### Same-Class Pairs

Our experiment trains with the pairing loss of that section, which compares a real sample with a generated one. We pair each real image with a generated image of the same class, a real bird with a generated bird. Within one class this is the pairing game between $p(\cdot \mid c)$ and $q(\cdot \mid c)$, with the objective $\Phi_c(D) = \mathbb{E}_{x \sim p(\cdot \mid c),\, x' \sim q(\cdot \mid c)}[\log \sigma(D(x, c) - D(x', c))]$. For one pair of distributions, that section showed that the value of the pairing game is the Jensen--Shannon divergence between the two orderings of a pair, real sample first ($p \otimes q$) or generated sample first ($q \otimes p$). The critic of each class can be optimized separately, so the conditional value averages these divergences over classes:

$$
\max_D \mathbb{E}_c\big[\Phi_c(D)\big] + \log 2
\;=\; \mathbb{E}_c\Big[\mathrm{JS}\big(p(\cdot \mid c) \otimes q(\cdot \mid c),\; q(\cdot \mid c) \otimes p(\cdot \mid c)\big)\Big].
$$

Each term is zero exactly when its slice is matched, so the fixed point is that of the equation. The shift invariance of the pairing game carries over class by class. Adding a constant to the critic leaves every score difference unchanged, and with same-class pairs this holds for one constant per class: $D(x, c) + \zeta_c$ has the same loss for any constants $\zeta_1, \ldots, \zeta_{10}$. The critic never calibrates its scores across classes, so our diagnostics must not depend on these constants (Exercise 3 measures them). The chapter's other results, such as the analysis of the zero-centered penalties, likewise hold class by class, since each concerns one pair of distributions.

## Conditioning Mechanisms

### Three Ways In

The analysis says what the optimal critic computes, but not how a network should take the condition in. Three mechanisms are common (the figure).

![Three ways a condition $c$ enters a GAN. Left: concatenation appends a class embedding $e_c$ to the latent code (or to an image or feature map in the critic). Middle: modulation scales and shifts a feature map $\phi$ by functions of the condition, $a(c) \odot \phi + b(c)$ (conditional batch normalization normalizes $\phi$ first). Right: the compatibility head of the projection critic scores the inner product of a class embedding with the critic features, $e_c^\top \phi(x)$, and adds an unconditional term $\psi_0(x)$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-gan-conditioning.svg)

*Concatenation* [@Mirza.Osindero.2014] embeds the condition in a vector $e_c$ and appends it to an input: beside the latent code in the generator, or beside the image or a feature map in the critic. It imposes no form on how sample and condition interact, so the network must learn that interaction from the adversarial signal alone. *Modulation* scales and shifts a feature map, $a(c) \odot \phi + b(c)$, with one entry per channel; BigGAN computes $a(c)$ and $b(c)$ from a class embedding in every block of its generator [@Brock.Donahue.Simonyan.2019]. Papers write $\gamma(c)$ and $\beta(c)$, but $\gamma$ is the penalty weight in this chapter. The third mechanism conditions only the critic, through a head that scores how well sample and condition fit together. Its most common form follows from the optimal critic.

### The Projection Head from Bayes' Rule

Bayes' rule, $p(x \mid c) = p(c \mid x)\, p(x) / p(c)$, applied to the numerator and the denominator of the equation, cancels the shared label marginal and yields

$$
\log \frac{p(x \mid c)}{q(x \mid c)}
\;=\; \log \frac{p(c \mid x)}{q(c \mid x)} + \log \frac{p(x)}{q(x)} .
$$

Thus the optimal conditional critic separates condition compatibility from unconditional sample quality. The second term is the log density ratio of the unconditional game. The first compares the real and generated *label posteriors* at $x$. A generator that produces realistic images with scrambled labels differs from the data through the first term; one that produces poorly formed images of the requested class differs through the second.

If both posteriors are softmax classifiers on one shared feature map $\phi$, the first term takes a familiar form. Let $p(c \mid x) \propto \exp(u_c^\top \phi(x))$ with one weight vector per class, and $q(c \mid x) \propto \exp(w_c^\top \phi(x))$ likewise. The two normalizers do not depend on $c$, so the log ratio of the posteriors is the inner product $(u_c - w_c)^\top \phi(x)$ plus a function of $x$ alone. Absorbing that function, together with the unconditional term, into a single head $\psi_0$ and writing $e_c = u_c - w_c$, the critic becomes

$$
D(x, c) \;=\; e_c^\top \phi(x) + \psi_0(x).
$$

This is the *projection discriminator* of @Miyato.Koyama.2018: one learned embedding per class, an inner product with the critic's features, and an unconditional head. Bayes' rule is an identity, but the shared feature map is a modeling assumption (Exercise 2). A linear $\psi_0$ can be folded into the class embeddings, and R3GAN's conditional critic keeps only the inner product [@Huang.Gokaslan.Kuleshov.ea.2024]; StyleGAN-T replaces the class embedding by the CLIP embedding of a caption, which makes the same head the text-conditioned critic of current one-step generators [@Sauer.Karras.Laine.ea.2023]. The *auxiliary classifier* critic of @Odena.Olah.Shlens.2017 is the main alternative: it adds a classification head, and the generator is trained to make its samples receive the requested label, which favors easily classified prototypes over within-class diversity (Exercise 7).

Our experiment uses concatenation in the generator and a projection head in the critic, the division R3GAN uses in its class-conditional experiments. The critic gets the structure that the derivation motivates; no such identity says how the generator should use $c$, so concatenation leaves that free.

### An Image as the Condition

The condition can also be an entire image: a semantic map, an edge sketch, or a grayscale photograph. Pix2pix [@Isola.Zhu.Zhou.ea.2017] plays the conditional game on output--input pairs, with the source image as $c$, and adds an $L_1$ loss against the paired target. The source fixes the coarse layout of the output, which the $L_1$ term recovers. Fine texture is ambiguous given the source, and the $L_1$ term alone blurs it, so pix2pix keeps the critic beside it.

## Class-Conditional CIFAR-10

We now train a class-conditional generator on the 50,000 labeled $32 \times 32$ training images of CIFAR-10, which come from ten classes and are loaded with `d2l.load_cifar10_arrays` (that section). R3GAN reports a class-conditional CIFAR-10 benchmark trained for four days on eight GPUs. Our run takes 15,000 steps, well under half an hour on one GPU, and tests the conditioning and its measurement, not large-scale sample quality.

In [ ]:
classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

In [ ]:
train_X, train_y, test_X, test_y = d2l.load_cifar10_arrays()
first = [int(np.nonzero(np.asarray(train_y) == c)[0][0]) for c in range(10)]
d2l.show_images(np.asarray(train_X[np.array(first)]) / 2 + 0.5,
                num_rows=2, num_cols=5, titles=classes);

### A Conditional Backbone

The networks are the saved `d2l.Generator` and `d2l.Discriminator` of that section at resolution 32, one resampling stage fewer than for the $64 \times 64$ sprites, with the condition as an extra input to each. Everything else is unchanged: bilinear resampling, leaky ReLU, no normalization, no tanh, and fan-in initialization. `CondGenerator` embeds the class, broadcasts the embedding over the $4 \times 4$ grid, and passes it as the base class's `extra` input, which is concatenated with the learned constant and the projected latent code before the mixing convolution.

In [ ]:
class CondGenerator(d2l.Generator):
    """The backbone generator with a class embedding as extra 4x4 maps."""
    def __init__(self, latent_dim, resolution, base_ch, embed_dim=64,
                 num_classes=10, *, rngs):
        super().__init__(latent_dim, resolution, base_ch, extra_ch=embed_dim,
                         rngs=rngs)
        self.embed = nnx.Embed(num_classes, embed_dim, rngs=rngs,
                               embedding_init=nnx.initializers.normal(1.0))

    def __call__(self, z, y):
        e = self.embed(y)[:, None, None, :]
        ec = jnp.broadcast_to(e, (z.shape[0], 4, 4, e.shape[-1]))
        return super().__call__(z, ec)

`ProjectionCritic` implements the equation. The base class's `phi` returns the $4 \times 4$ feature map after the mixing convolution; summing it over positions gives $\phi(x)$. A linear head on $\phi(x)$ gives $\psi_0(x)$ and replaces the base head, which reads the flattened map, and a table holds one embedding $e_c$ per class. The table starts at zero, so the critic begins as an unconditional critic and learns the label term from the data.

In [ ]:
class ProjectionCritic(d2l.Discriminator):
    """The backbone critic with a projection head: e_c . phi(x) + psi_0(x)."""
    def __init__(self, resolution, base_ch, num_classes=10, *, rngs):
        super().__init__(resolution, base_ch, rngs=rngs)
        width = self.mix.out_features
        self.head = nnx.Linear(width, 1, kernel_init=d2l.fan_in_init,
                               rngs=rngs)  # psi_0 on the pooled features
        self.embed = nnx.Embed(num_classes, width, rngs=rngs,
                               embedding_init=nnx.initializers.zeros)

    def __call__(self, x, y):
        phi = self.phi(x).sum(axis=(1, 2))  # sum over the 4x4 positions
        return (self.embed(y) * phi).sum(axis=1) + self.head(phi).squeeze(-1)

net_G = CondGenerator(100, 32, 256, rngs=nnx.Rngs(0))
net_D = ProjectionCritic(32, 64, rngs=nnx.Rngs(0))
print(net_G(jnp.zeros((2, 100)), jnp.zeros(2, dtype=jnp.int32)).shape)
print(net_D(jnp.zeros((2, 32, 32, 3)), jnp.zeros(2, dtype=jnp.int32)).shape)

### Training with Same-Class Pairs

Training follows the sprite recipe of that section and runs its saved loop `d2l.train_r3gan`: the pairing loss for both players, the penalties $R_1$ and $R_2$ on real and generated images, Adam with $\beta_1 = 0$, random horizontal flips of the real images, and an exponential moving average of the generator (`d2l.EMA`, half-life 500 steps) that supplies the generator for evaluation. Same-class pairs come from the labels: given them, the loop draws a labeled real batch and conditions the generated batch on the same labels. With the labels as `y`, the saved steps `d2l.critic_step` and `d2l.generator_step` of that section score with the critic $x \mapsto D(x, y)$, so the losses and the penalty see an ordinary critic of images.

With the test images as its evaluation images, the ranking accuracy that the loop logs every 250 steps becomes the *held-out pair accuracy*: on 512 fresh pairs of a test image and a generated image of the same class, the fraction that the critic orders correctly, $\hat P(D(x, c) > D(x', c))$. The constants $\zeta_c$ cancel from it, whereas the critic's mean score drifts with them. A value of one half means that the critic cannot tell real from generated images of a class.

The runs use the penalty weight $\gamma = 0.5$, against $\gamma = 10$ for the sprites. The penalty weight is a squared length on the scale of the input (that section), and the right value also depends on the critic, the batch size and the budget, so a new setup needs its own sweep (Exercise 8).

In [ ]:
rngs = nnx.Rngs(0)  # the initialization
net_G = CondGenerator(100, 32, 256, rngs=rngs)
net_D = ProjectionCritic(32, 64, rngs=rngs)
gamma = 0.5
ema_G, history = d2l.train_r3gan(
    net_G, net_D, d2l.rpgan_loss_D, d2l.rpgan_loss_G, gamma, 15000, train_X,
    train_y, eval_images=test_X, eval_labels=test_y, eval_size=512,
    batch_size=128, seed=1)

We average the logs over the last 5,000 steps.

In [ ]:
loss_D, loss_G, pen, acc = (np.mean(v[-5000 // 250:]) for v in (
    history['loss_D'], history['loss_G'], history['R1'] + history['R2'],
    history['accuracy']))
print(f'critic loss {loss_D:.2f} (+ penalty {gamma / 2 * pen:.3f}), '
      f'generator loss {loss_G:.2f}, pair accuracy {acc:.2f}')

In [ ]:
steps = history['step']
fig, axes = d2l.plt.subplots(1, 2, figsize=(9, 3.2))
d2l.plot(steps, [history['loss_D'], history['loss_G'],
                 np.full_like(steps, np.log(2))], 'step', 'loss',
         legend=['critic (pairing part)', 'generator', 'log 2'],
         fmts=('-', '--', 'k:'), yscale='log', axes=axes[0])
d2l.plot(steps, [history['accuracy'], np.full_like(steps, 0.5)], 'step',
         'held-out pair accuracy', ylim=[0.3, 1], fmts=('-', 'k:'),
         axes=axes[1]);

The dotted lines mark $\log 2$, the value of either pairing loss when the critic cannot order a pair, and a pair accuracy of one half. Both losses settle within the first few thousand steps, after a high start of the generator loss: the generator's near 1.0, above $\log 2$, and the critic's near 0.5, below it, with the penalty adding about 0.06. The critic still wins most comparisons: late in training it orders about 80% of fresh same-class pairs correctly, down from about 90% early in training. At this budget the generator has not caught up with the data.

### Samples on Request

Each panel of the next figure requests one class, and its nine images use fresh latent codes; `d2l.tile_images` of that section tiles them. As in that section, the generator has no tanh, so the display and every evaluation below clamp its output to $[-1, 1]$.

In [ ]:
z = jax.random.normal(jax.random.key(42), (90, 100))
y = jnp.repeat(jnp.arange(10), 9)  # nine per class
imgs = ema_G(z, y)
panels = [d2l.tile_images(imgs[9 * c:9 * (c + 1)], 3) for c in range(10)]
d2l.show_images(panels, 2, 5, titles=classes, scale=2);

Most panels read as their class. Vehicles are recognizable by shape: cars and trucks as wheeled bodies, ships as hulls on water, airplanes against the sky. The animal panels lean more on palette and background: frogs are green and brown textures, deer and horses brown forms on grass, while birds, cats and dogs are often similar brown shapes. Pose, color and background vary within every panel, so the figure shows no obvious collapse within a class. Inspection cannot say how often a generated image would be recognized as its class, or how much variety is missing; we measure both next.

## Measuring Condition Alignment

### A Referee Classifier

An unconditional metric cannot detect the cyclic label shift, which preserves the distribution of images while violating every request. We therefore check requests with a CIFAR-10 classifier, the *referee*. It is the feature network of that section, trained the same way: `d2l.train_feature_cnn` at CIFAR-10's native $32 \times 32$ resolution with random horizontal flips, random shifts of up to four pixels, and a learning rate that rises and then anneals over 15 epochs. Without the augmentation and the schedule the network overfits, and its accuracy on a single class moves by tens of points between two training seeds, more than the differences we want to read. With them it reaches about 78%, and its per-class accuracies move by a few points.

In [ ]:
cnn = d2l.FeatureCNN(resolution=32, rngs=nnx.Rngs(1))
d2l.train_feature_cnn(cnn, train_X, train_y, shift=4)
pred = d2l.batched(cnn, test_X).argmax(1)
cls_acc = [float((pred[test_y == c] == c).mean()) for c in range(10)]
print(f'CIFAR-10 test accuracy: {float((pred == test_y).mean()):.3f}')

### Alignment per Class

For each generated image, the referee either predicts the requested class or it does not. The fraction of matches, overall and per class, is the *condition alignment*. We compare it with the referee's accuracy on real test images of the same class, because the referee also misclassifies real images. The real accuracy is a reference, not an upper bound: unusually prototypical generated images can be easier to classify than photographs. The next cell generates 500 images of each class and the one after it prints both and their difference, the *gap*.

In [ ]:
y_req = jnp.repeat(jnp.arange(10), 500)  # requests
z = jax.random.normal(jax.random.key(7), (5000, 100))
pred_req = cnn(jnp.clip(ema_G(z, y_req), -1, 1)).argmax(1)
align = [float((pred_req[y_req == c] == c).mean()) for c in range(10)]

In [ ]:
print(f'{"class":12s}{"alignment":>11s}{"real":>7s}{"gap":>7s}')
for name, a, r in zip(classes + ['mean'], align + [np.mean(align)],
                      cls_acc + [np.mean(cls_acc)]):
    print(f'{name:12s}{a:11.2f}{r:7.2f}{a - r:+7.2f}')

Averaged over classes, alignment is about 0.7, six or seven points below the referee's accuracy on real images. No class is recognized much more often than its real counterpart, and no class is harder than cat in either column. Individual gaps move by several points between runs, so the table does not rank them.

### Fidelity and Within-Class Diversity

Alignment does not measure image quality. We therefore also compute the two feature-distribution distances of that section: the Fréchet distance between Gaussian fits to two feature sets, and the unbiased $\mathrm{MMD}^2$ estimate with a cubic polynomial kernel. Both use the referee's 128-dimensional features, so we call them FD and $\mathrm{MMD}^2$ (CIFAR-CNN); they cannot be compared with published FID or KID scores. The evaluation compares 1,000 generated images, 100 per class, with 1,000 test images in the same proportions, and a second disjoint set of 1,000 test images gives the real-versus-real floor. The distances ignore the labels, so the table is followed by the *within-class feature variance*: the variance of the features within each class, averaged over features and classes.

In [ ]:
features = lambda imgs: np.asarray(d2l.batched(cnn.features, imgs))
yg = jnp.repeat(jnp.arange(10), 100)
raw = ema_G(jax.random.normal(jax.random.key(11), (1000, 100)), yg)
out_of_range = float((jnp.abs(raw) > 1).mean())
gen = jnp.clip(raw, -1, 1)
idx = [jnp.nonzero(test_y == c)[0][:200] for c in range(10)]
f_real = features(test_X[jnp.concatenate([i[:100] for i in idx])])
f_real2 = features(test_X[jnp.concatenate([i[100:] for i in idx])])
f_gen = features(gen)

In [ ]:
print(f'{"comparison":22s}{"FD (CIFAR-CNN)":>16s}{"MMD^2 (CIFAR-CNN)":>19s}'
      f'{"out-of-range":>14s}')
print(f'{"real vs. real":22s}{d2l.fid_score(f_real2, f_real):16.2f}'
      f'{d2l.kid_score(f_real2, f_real):19.3f}{"--":>14s}')
print(f'{"generated vs. real":22s}{d2l.fid_score(f_gen, f_real):16.2f}'
      f'{d2l.kid_score(f_gen, f_real):19.3f}{out_of_range:14.3f}')
# Variance within each class, averaged over features and classes
cvar = lambda f: float(np.asarray(f, np.float64).reshape(10, 100, -1)
                       .var(1).mean())
print(f'within-class feature variance: generated {cvar(f_gen):.3g}, '
      f'real {cvar(f_real):.3g}, second real set {cvar(f_real2):.3g}')

In FD the generated set lies two to four times as far from the real test images as the second real set does. In $\mathrm{MMD}^2$ the real-versus-real estimate fluctuates around zero, while the generated-versus-real estimate is small and positive. The variance line shows what the distances cannot separate: the generated images have about 20% less within-class feature variance than the real ones, while a second real set comes within 5%. The generator has lost some of the variety within each class, and alignment cannot see that loss.

### Two Baselines

Two constructed generators show what each measurement can and cannot see. The first keeps the same 1,000 generated images and shifts their *requested* labels cyclically, the counterexample from the start of this section. The second is a degenerate generator that returns, for every request of class $c$, the real test image that the referee classifies as $c$ most confidently. FD and $\mathrm{MMD}^2$ never see a label, so for the first baseline the values of the previous cell stand; we print only the alignment.

In [ ]:
y_perm = (yg + 1) % 10                         # same images, shifted requests
balanced = jnp.concatenate([i[:100] for i in idx])  # 100 test images per class
pred_gen = cnn(gen).argmax(1)
# The referee's probability of the true class, per class and image
p_true = jax.nn.softmax(cnn(test_X[balanced]))[
    jnp.arange(1000), test_y[balanced]].reshape(10, 100)
most_confident = balanced.reshape(10, 100)[jnp.arange(10), p_true.argmax(1)]
fixed = jnp.repeat(test_X[most_confident], 100, axis=0)
pred_fix = cnn(fixed).argmax(1)
print(f'permuted labels: alignment '
      f'{float((pred_gen == yg).mean()):.2f} -> '
      f'{float((pred_gen == y_perm).mean()):.2f}')
print(f'most confident image per class: alignment '
      f'{float((pred_fix == yg).mean()):.2f}, '
      f'within-class variance {cvar(features(fixed)):.3g}')

Shifting the requests drops alignment from about 0.7 to a few percent, while FD and $\mathrm{MMD}^2$ cannot change, because neither receives a label. The confident-image generator reaches alignment 1.00 with zero within-class variance: it beats real images on alignment while returning one image per request. This *within-class collapse* is mode collapse inside each conditional distribution. The conditional objective the equation penalizes it, because a point mass is far from $p(\cdot \mid c)$, and so does the within-class variance; alignment rewards it.

Conditional evaluation thus has three axes, fidelity, diversity and alignment, and each of our measurements sees only part of them. Published scores combine them: the Fréchet joint distance embeds image and condition together [@DeVries.Romero.Pineda.ea.2019], and the classification accuracy score trains a classifier on generated pairs and tests it on real ones [@Ravuri.Vinyals.2019]. Exercise 4 extends our diagnosis to per-class distances, and Exercise 5 uses the same instruments on BigGAN's truncation trick, which trades variety for fidelity on purpose.

We can now ask a GAN for a class and check what we got: whether it obeys, how much it varies, and how close it comes to the data. Pix2pix has a paired target for every input and still needs its critic. Why a learned comparison helps when a regression target exists is the question of that section.

## Summary

Conditional generation gives the request to both players and plays the chapter's game on sample--condition pairs. Its optimal critic is the per-class log density ratio $\log(p(x \mid c)/q(x \mid c))$ and its value averages the per-class divergences, so it detects perfect images with the wrong labels, which the unconditional game cannot. With same-class pairs the pairing loss plays the game of that section within every class, up to one free constant per class. Bayes' rule splits the optimal critic into a label term and a realness term, which gives the projection critic $D(x, c) = e_c^\top \phi(x) + \psi_0(x)$. On CIFAR-10 the referee recognizes the requested class in about 70% of the generated images, against 78% of real images, and the generated images have about a fifth less within-class variety than real data. Alignment, feature distances and within-class variance each catch a different failure.

## Exercises

1. [code] **Two slices with known densities.** Let $c \in \{1, 2\}$ be
   equally likely, $p(\cdot \mid 1) = \mathcal{N}(-2, 1)$ and
   $p(\cdot \mid 2) = \mathcal{N}(2, 1)$, and let the generator be wrong on
   one slice only: $q(\cdot \mid 1) = \mathcal{N}(0, 1)$ and
   $q(\cdot \mid 2) = \mathcal{N}(2, 1)$.
    1. Compute the value the equation by numerical
       integration on a grid.
    1. Train a small critic on $x$ and a one-hot encoding of $c$ by
       minimizing $-\tfrac12 V(D)$ of the equation on samples, and
       compare the converged loss with $-\tfrac12$ times the value from (a).
       Plot the critic against the equation for each
       condition. How large is it on the matched slice $c = 2$?
    1. Let the generator swap the labels instead, $q(\cdot \mid 1) =
       \mathcal{N}(2, 1)$ and $q(\cdot \mid 2) = \mathcal{N}(-2, 1)$.
       Compute the conditional value and the unconditional value with the
       labels discarded, and check both by retraining the critic.
1. **Separate feature maps.** The derivation of
   the equation modeled both label posteriors with one
   shared feature map $\phi$. Suppose instead that
   $p(c \mid x) \propto \exp(u_c^\top \phi_p(x))$ and
   $q(c \mid x) \propto \exp(w_c^\top \phi_q(x))$ with
   $\phi_p \neq \phi_q$.
    1. Derive the critic that takes the place of
       the equation, and show that it is a projection
       head over the concatenated feature map $(\phi_p(x), \phi_q(x))$.
    1. Let there be $K$ classes. Show that every function $D(x, c)$ can be
       written in the form the equation once $\phi$
       has at least $K$ dimensions. How many dimensions does $\phi$ have
       in `ProjectionCritic`?
    1. Given (b), when does the projection form restrict the set of
       critics? What does the shared-map assumption then restrict in the
       CIFAR-10 experiment?
1. [short-code] **The critic as a classifier.** The trained critic `net_D`
   assigns each image ten scores, one per class.
    1. Use the equation to express
       $D^\star(x, c) - D^\star(x, c')$ through the real and the generated
       label posteriors. How does $\arg\max_c D^\star(x, c)$ differ from
       the rule $\arg\max_c p(c \mid x)$ of the referee?
    1. For the test images indexed by `bal`, compute $S_{ic} = D(x_i, c)$
       for all ten classes. Report the fraction of images that score higher
       under their own label $c$ than under $c + 1$, and the accuracy of
       $\arg\max_c S_{ic}$. Then estimate the per-class constant
       $\hat\zeta_c$ as the critic's mean score on the generated images
       `gen` of class $c$ (labels `yg`), and classify with
       $\arg\max_c (S_{ic} - \hat\zeta_c)$. Expect between a quarter and a
       third of the images to be classified correctly, two to three times
       chance. Why does the raw rule do worse?
    1. Why must the corrected rule fall to chance as the generator
       improves? If the first fraction in (b) fell to one half after longer
       training, name two states of the game that would produce this, and a
       measurement of this section that tells them apart.
1. [code] **Per-class distances.** The features `f_gen`, `f_real` and
   `f_real2` hold 100 images of each class, in class order.
    1. A covariance matrix estimated from 100 samples of 128-dimensional
       features has rank at most 99. How does this affect a per-class
       Fréchet distance, and why does the unbiased $\mathrm{MMD}^2$
       estimator not suffer in the same way?
    1. For each class, compute $\mathrm{MMD}^2$ between the generated and
       the real features, and between the two real sets. Rank the classes by
       the difference and compare with the ranking by alignment; display the
       samples of a class on which the two rankings disagree.
    1. A conditional generator may ignore the condition, obey it through
       shortcuts such as background or palette, collapse within a condition,
       leak between conditions, or neglect a rare condition. Which of these
       failures does each per-class diagnostic (alignment, feature variance,
       $\mathrm{MMD}^2$) detect? Check your answer on the collapse baseline.
1. [code] **Truncated latents.** The truncation trick of
   @Brock.Donahue.Simonyan.2019 draws each latent coordinate from a
   standard Gaussian truncated to $[-\tau, \tau]$. Such a coordinate has
   variance
   $1 - \sqrt{2/\pi}\,\tau e^{-\tau^2/2} / \operatorname{erf}(\tau/\sqrt{2})$.
    1. For the 100-dimensional latent code of `CondGenerator` and
       $\tau \in \{0.5, 1, 2\}$, compute the probability that an untruncated
       draw lies inside $[-\tau, \tau]^{100}$ and the typical norm of a
       truncated draw. Compare it with the norm of the training latents.
    1. Implement truncated sampling for `ema_G` and sweep
       $\tau \in \{0.5, 1, 2, \infty\}$, where $\tau = \infty$ is ordinary
       sampling. Report the alignment, the within-class feature variance and
       the Fréchet distance to `f_real`, and display the samples.
    1. BigGAN found that only 16% of its models trained without orthogonal
       regularization tolerated truncation without saturation artifacts,
       against 60% with it; `ema_G` had none. At which $\tau$ do your
       measurements change most, given the norms from (a)? What does this
       say about how the generator treats latent codes with norms it never
       saw in training?

    *Adapted from Simon Prince,
    [Understanding Deep Learning](https://udlbook.github.io/udlbook/),
    Problem 15.6.*
1. [code] **A rare class.** ● Keep only 50 of the 5,000 automobile training
   images and all images of the other classes, so that
   $p(c_{\textrm{rare}}) = \epsilon = 50/45{,}050 \approx 0.0011$.
    1. Show that a generator that serves the rare slice arbitrarily badly
       raises the equation by at most $2 \epsilon \log 2$.
       Class-balanced sampling of the real batches raises the weight of the
       slice to $1/10$. Which risk does it introduce, given that the slice
       holds only 50 images?
    1. Train with `d2l.train_r3gan` for 4,000 steps on real batches drawn
       uniformly from the reduced set, and on class-balanced real batches,
       whose labels then condition the generator uniformly; as a reference,
       train at the same budget on the full set. For the rare class, report
       the alignment, the per-class $\mathrm{MMD}^2$ of problem
       4b, and the
       feature distance of each generated sample to its nearest neighbor
       among the 50 training images. Which scheme serves the rare class
       better, and what does it cost the other classes?
1. **Auxiliary classifier bias.** ● An auxiliary classifier critic
   [@Odena.Olah.Shlens.2017] contains a classifier $C(c \mid x)$, and
   the generator also minimizes the classification loss
   $\mathbb{E}_{c \sim p(c),\, x \sim q(\cdot \mid c)}[-\log C(c \mid x)]$
   of its own samples. Let $q(x)$ and $q(c \mid x)$ be the image marginal
   and the label posterior of the generated pairs.
    1. Show that this loss equals

       $$\mathbb{E}_{x \sim q}\Big[H\big(q(\cdot \mid x)\big) + D_{\mathrm{KL}}\big(q(\cdot \mid x) \,\|\, C(\cdot \mid x)\big)\Big],$$

       where $H$ is the entropy.
    1. Hold $C$ fixed, set $q(x) = p(x)$, and let $q(\cdot \mid x)$ vary
       freely at every $x$, ignoring the constraint that the generated labels
       follow $p(c)$. Which label posterior minimizes the loss? With
       $C = p(c \mid x)$ and two overlapping classes, show that the
       generator can lower the loss below its value at $q = p$.
    1. Take the expectation of the label term
       $\log [p(c \mid x) / q(c \mid x)]$ of the equation
       under the generated pairs. Which term of the loss in (a) does its
       denominator cancel?
1. [code] **The critic at a fixed budget.** ● The run above fixes the
   penalty weight $\gamma = 0.5$ and the projection head.
    1. Train for 4,000 steps at $\gamma \in \{0.05, 0.5, 5\}$, and report
       the logged squared input gradients (`history['R1'] + history['R2']`), the
       alignment,
       and the within-class feature variance. Which change by an order of
       magnitude, and which barely move? Since $\gamma$ is a squared length
       in pixel space, why does CIFAR-10 need a different value from the
       sprites?
    1. Replace the projection head of `ProjectionCritic` by concatenation:
       append the one-hot label to the image as ten constant
       $32 \times 32$ planes and score with the head $\psi_0$ alone. Train
       both critics with `d2l.train_r3gan` for 4,000 steps with two seeds
       each. Compare the alignment,
       the feature distances, and the critic's accuracy as a classifier with
       its constants removed (problem 3b).
    1. By problem
       2b, neither
       critic restricts the function $D(x, c)$ for ten classes. What can
       then explain a difference at a fixed budget?

[Discussions](https://d2l.discourse.group/t/36813)